# ICOS atmospheric Level 2 — two species at once

**Experimental variant.** The standard notebook combines the data objects of *one* data type — in
practice the several measurement heights of one species at one station. This one instead loads
**two data types** for the same station, so that two species can be plotted together: CO against
CO₂, for instance.

Everything else — configuration, authentication, station choice — works exactly as in
`ICOS_ex1a_atmo_data_L2.ipynb`.

> **Clearing the outputs.** Use *Edit → Clear Outputs of All Cells* before running the notebook
> again and before saving it: widget state belongs to the kernel that created it, and leftovers
> make the controls stop responding.

## 0. Configuration

In [ ]:
# The kernel keeps a handle on the directory it was started in. If that directory is replaced
# while the kernel is running - the shared workspace being republished, for instance - every
# relative path fails with "Stale file handle" or FileNotFoundError. Recover instead of leaving
# the user with an error that has nothing to do with what they were doing.
import os

try:
    os.getcwd()
except OSError:
    _nb = os.environ.get("JPY_SESSION_NAME", "")
    _target = os.path.dirname(_nb) if _nb else os.path.expanduser("~")
    if not os.path.isdir(_target):
        _target = os.path.expanduser("~")
    os.chdir(_target)
    print(f"The working directory had become stale and was reset to {_target}.")
    print("If anything still looks wrong, restart the kernel (Kernel -> Restart Kernel).")

# Identity of this notebook
NOTEBOOK_NAME = "ICOS atmospheric Level 2 - two species"
NOTEBOOK_VERSION = "0.1"
NOTEBOOK_DATE = "2026-09-15"

DEFAULT_COUNTRY = "IT"
DEFAULT_STATION_CODE = "CMN"
# The two data types the notebook opens on, matched as substrings against the data-type label.
# They have to be specific: "ATC CO" would also match "ICOS ATC CO2 Release".
DEFAULT_DATA_TYPES = ["ATC CO Release", "ATC CO2 Release"]

# Backend used for the station map, as in the standard notebook: "plotly" or "matplotlib".
MAP_BACKEND = "plotly"

DEFAULT_QUALITY_FLAG = "Flag"
VALUE_ACCEPT_QUALITY = "O"
MAX_OBJECTS_PER_TYPE = 20
# When the two products do not share exact timestamps, each point is matched with the nearest
# one within this window instead of being discarded.
ALIGN_TOLERANCE = "30min"
TIME_COLUMN_CANDIDATES = ("SamplingStart", "TIMESTAMP", "TimeStamp", "time", "Date")

ICOS_COOKIE_TOKEN = ""

print(f"{NOTEBOOK_NAME} - version {NOTEBOOK_VERSION} ({NOTEBOOK_DATE})")

## 1. Dependencies and imports

In [ ]:
import os
import importlib.util
import math
import subprocess
import sys

# Install only what is missing. Each VRE user has their own container, so a notebook that runs
# for a colleague can still fail here with ModuleNotFoundError.
REQUIRED = {
    "icoscp_core": "icoscp",
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "plotly": "plotly",
    "ipywidgets": "ipywidgets",
    # Hidden dependency: plotly needs nbformat to render figures but does not declare it.
    "nbformat": "nbformat",
}
missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print(f"Installing: {', '.join(missing)}")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing],
                       capture_output=True, text=True,
                            cwd=os.path.expanduser("~"))
    print("Done." if r.returncode == 0 else r.stderr[-1500:])
else:
    print("All required packages are already installed.")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import plotly.graph_objects as go
from plotly.offline import init_notebook_mode
import ipywidgets as widgets
from IPython.display import display

from icoscp_core.icos import data, meta, ATMO_STATION

# Loads the Plotly JavaScript into the page. Without it, figures shown inside an ipywidgets
# Output (the station map) render as an empty area. connected=True pulls it from the CDN
# instead of embedding several MB into the saved notebook.
if MAP_BACKEND == "plotly":
    init_notebook_mode(connected=True)


# --- Outputs of this run -------------------------------------------------------------
# Figures are shown inline and kept in memory; nothing is written to disk while you work.
# The export cell at the end of the notebook writes out what you decide to keep, into the
# folder below, and packs it into a single zip you can download.
#
# The folder is emptied at the start of each run, so that repeated runs do not accumulate
# files. Set CLEAN_OUTPUTS to False to keep what earlier runs produced.
import shutil
from pathlib import Path

OUTPUT_DIR = Path("outputs")
CLEAN_OUTPUTS = True

if CLEAN_OUTPUTS and OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
    if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
        print("Note: some files could not be removed (still open elsewhere); continuing anyway.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DRAWN_FIGURES = {}


def register_figure(fig, stem):
    """Remember a figure under a name, for the export cell at the end."""
    DRAWN_FIGURES[stem] = fig

# Archives produced by the export cell at the end go here. The folder is emptied together with
# the rest, so the zips of previous runs do not pile up next to the notebook.
EXPORT_DIR = Path("exports")
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR, ignore_errors=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
from icoscp_core.icos import auth, bootstrap


def _authenticated() -> bool:
    """auth.get_token() raises when no usable credentials are stored, so it doubles as a probe."""
    try:
        auth.get_token()
        return True
    except Exception:
        return False


if ICOS_COOKIE_TOKEN.strip():
    # Option B. bootstrap.fromCookieToken returns *new* metadata and data clients: they have to
    # replace the ones imported in section 1, otherwise the rest of the notebook would keep
    # using the unauthenticated connection and fail later, far from the cause.
    try:
        meta, data = bootstrap.fromCookieToken(ICOS_COOKIE_TOKEN.strip())
        meta.list_stations(ATMO_STATION)      # cheap call, fails fast if the token is invalid
        print("Authenticated with the token from the Configuration cell.")
        print("Remember: it expires in under 28 hours, and must not be left in a shared copy "
              "of this notebook.")
    except Exception as exc:  # noqa: BLE001 - an expired or mistyped token must be readable
        raise RuntimeError(
            f"The token in ICOS_COOKIE_TOKEN was not accepted ({type(exc).__name__}: {exc}).\n"
            "Tokens expire in under 28 hours - fetch a fresh one from the 'My Account' page at "
            "https://cpauth.icos-cp.eu/, or clear ICOS_COOKIE_TOKEN to use option A instead."
        ) from exc

elif _authenticated():
    print("Already authenticated with stored credentials - nothing to do.")

else:
    # Option A: prompts once, then stores the configuration in the home directory
    try:
        auth.init_config_file()
    except Exception as exc:  # noqa: BLE001 - authentication failures should be readable
        print(f"Authentication setup failed ({type(exc).__name__}: {exc})")

    if _authenticated():
        print("\nAuthentication configured - move on to the next section.")
    else:
        print("\nStill not authenticated. Either set a password for your Carbon Portal account "
              "at https://cpauth.icos-cp.eu/ and re-run this cell, or use a token (option B "
              "above) by setting ICOS_COOKIE_TOKEN in the Configuration cell.")

## 2. Authentication

Accessing ICOS data through the `icoscp` library requires authentication, which is how the Carbon
Portal records that you have accepted the data licence. You need an account at
[cpauth.icos-cp.eu](https://cpauth.icos-cp.eu/) and you must accept the ICOS Data Licence in your
user profile. There are two ways to authenticate.

### Option A — Carbon Portal credentials (default, persistent)

The cell below asks for your Carbon Portal email and password and stores the configuration in your
home directory, so you are only asked once. This requires an account whose password the Carbon
Portal itself can verify: if you normally sign in through a federated provider (ORCID, institutional
account) and have never set a password, either set one in your profile, or use option B.

### Option B — authentication token (works with any login method, temporary)

A token works regardless of how you sign in, which makes it the way in for federated accounts. Log
in at [cpauth.icos-cp.eu](https://cpauth.icos-cp.eu/) with whichever mechanism you use, open **My
Account**, copy the token, and set it in the Configuration cell:

```python
ICOS_COOKIE_TOKEN = "cpauthToken=<token from My Account>"
```

Two caveats. The token expires after **less than 28 hours**, so it suits a working session rather
than a permanent setup. And it is written in the notebook source: **do not leave it in a notebook
you copy back into the shared workspace**, where every VRE user could read and reuse it until it
expires. Keep it in your own copy in your home directory and clear the variable before sharing.

Either way, this cell is safe to re-run: if valid credentials are already available it says so and
asks for nothing.

In [ ]:
from icoscp_core.icos import auth, bootstrap


def _authenticated() -> bool:
    """auth.get_token() raises when no usable credentials are stored, so it doubles as a probe."""
    try:
        auth.get_token()
        return True
    except Exception:
        return False


if ICOS_COOKIE_TOKEN.strip():
    # Option B. bootstrap.fromCookieToken returns *new* metadata and data clients: they have to
    # replace the ones imported in section 1, otherwise the rest of the notebook would keep
    # using the unauthenticated connection and fail later, far from the cause.
    try:
        meta, data = bootstrap.fromCookieToken(ICOS_COOKIE_TOKEN.strip())
        meta.list_stations(ATMO_STATION)      # cheap call, fails fast if the token is invalid
        print("Authenticated with the token from the Configuration cell.")
        print("Remember: it expires in under 28 hours, and must not be left in a shared copy "
              "of this notebook.")
    except Exception as exc:  # noqa: BLE001 - an expired or mistyped token must be readable
        raise RuntimeError(
            f"The token in ICOS_COOKIE_TOKEN was not accepted ({type(exc).__name__}: {exc}).\n"
            "Tokens expire in under 28 hours - fetch a fresh one from the 'My Account' page at "
            "https://cpauth.icos-cp.eu/, or clear ICOS_COOKIE_TOKEN to use option A instead."
        ) from exc

elif _authenticated():
    print("Already authenticated with stored credentials - nothing to do.")

else:
    # Option A: prompts once, then stores the configuration in the home directory
    try:
        auth.init_config_file()
    except Exception as exc:  # noqa: BLE001 - authentication failures should be readable
        print(f"Authentication setup failed ({type(exc).__name__}: {exc})")

    if _authenticated():
        print("\nAuthentication configured - move on to the next section.")
    else:
        print("\nStill not authenticated. Either set a password for your Carbon Portal account "
              "at https://cpauth.icos-cp.eu/ and re-run this cell, or use a token (option B "
              "above) by setting ICOS_COOKIE_TOKEN in the Configuration cell.")

## 3. Choose the station

The same station list as the standard notebook: pick a country, then a station.

In [ ]:
stations = meta.list_stations(ATMO_STATION)
stations_by_code = {s.id: s for s in stations}
print(f"{len(stations)} ATMO stations available.")


def _country_options():
    codes = sorted({s.country_code for s in stations if s.country_code})
    return [("All countries", "ALL")] + [(c, c) for c in codes]


def _stations_for(country):
    sel = [s for s in stations if country == "ALL" or s.country_code == country]
    return sorted((f"{s.id} - {s.name} ({s.country_code})", s.id) for s in sel)


if "SELECTED_STATION" not in globals():
    SELECTED_STATION = None
_initial = SELECTED_STATION if SELECTED_STATION in stations_by_code else DEFAULT_STATION_CODE
if _initial not in stations_by_code:
    _initial = stations[0].id

country_dd = widgets.Dropdown(options=_country_options(),
                              value=stations_by_code[_initial].country_code or "ALL",
                              description="Country:", style={"description_width": "initial"},
                              layout=widgets.Layout(width="320px"))
station_dd = widgets.Dropdown(options=_stations_for(country_dd.value), value=_initial,
                              description="Station:", style={"description_width": "initial"},
                              layout=widgets.Layout(width="520px"))
confirm_station_btn = widgets.Button(description="Confirm station", button_style="success",
                                     icon="check")
station_out = widgets.Output()


def _on_country(change):
    opts = _stations_for(change["new"])
    station_dd.options = opts
    if opts:
        station_dd.value = opts[0][1]


def _on_station_confirm(_):
    global SELECTED_STATION
    SELECTED_STATION = station_dd.value
    with station_out:
        station_out.clear_output()
        s = stations_by_code[SELECTED_STATION]
        print(f"Confirmed: {s.id} - {s.name}")


country_dd.observe(_on_country, names="value")
confirm_station_btn.on_click(_on_station_confirm)
display(widgets.VBox([country_dd, station_dd, confirm_station_btn, station_out]))

with station_out:
    print(f"Will default to {DEFAULT_STATION_CODE} unless you confirm another station.")

## 4. Choose the two data types

Pick exactly two: one per axis of the plot further down. They are listed as the standard notebook
lists them — Level 2, atmospheric, with data access.

In [ ]:
all_data_types = meta.list_datatypes()
selected_datatypes = [
    dt for dt in all_data_types
    if dt.has_data_access and dt.data_level == 2
    and dt.theme.uri == "http://meta.icos-cp.eu/resources/themes/atmosphere"
]
datatypes_by_uri = {dt.uri: dt for dt in selected_datatypes}
_dt_options = sorted((dt.label, dt.uri) for dt in selected_datatypes)
print(f"{len(selected_datatypes)} previewable Level 2 atmospheric data type(s).")


def _match_default(keyword):
    """First data type whose label contains the keyword, or None."""
    for label, uri in _dt_options:
        if keyword.lower() in label.lower():
            return uri
    return None


if "SELECTED_TYPES" not in globals():
    SELECTED_TYPES = None
_defaults = [u for u in (_match_default(k) for k in DEFAULT_DATA_TYPES) if u]

dt_a = widgets.Dropdown(options=_dt_options,
                        value=_defaults[0] if _defaults else _dt_options[0][1],
                        description="First:", style={"description_width": "initial"},
                        layout=widgets.Layout(width="760px"))
dt_b = widgets.Dropdown(options=_dt_options,
                        value=_defaults[1] if len(_defaults) > 1 else _dt_options[-1][1],
                        description="Second:", style={"description_width": "initial"},
                        layout=widgets.Layout(width="760px"))
confirm_types_btn = widgets.Button(description="Confirm data types", button_style="success",
                                   icon="check")
types_out = widgets.Output()


def _on_types_confirm(_):
    global SELECTED_TYPES
    with types_out:
        types_out.clear_output()
        if dt_a.value == dt_b.value:
            print("The two data types are the same - pick two different ones.")
            SELECTED_TYPES = None
            return
        SELECTED_TYPES = [dt_a.value, dt_b.value]
        for uri in SELECTED_TYPES:
            print(f"  {datatypes_by_uri[uri].label}")


confirm_types_btn.on_click(_on_types_confirm)
display(widgets.VBox([dt_a, dt_b, confirm_types_btn, types_out]))

with types_out:
    print("Not confirmed yet - will default to:")
    for uri in (_defaults or [_dt_options[0][1]]):
        print(f"  {datatypes_by_uri[uri].label}")

## 5. Load both and merge on time

For each of the two data types, every data object of the selected station is read and its numeric
columns are kept, renamed with the station code and the measurement height so that two species —
or the same species at two heights — never collide.

The two are then merged on the time column. Only timestamps present in both survive, which is what
makes a comparison meaningful: if the merge comes back empty, the two records do not overlap in
time.

In [ ]:
SELECTED_STATION = globals().get("SELECTED_STATION") or DEFAULT_STATION_CODE
SELECTED_TYPES = globals().get("SELECTED_TYPES") or [u for u in
                                                     (_match_default(k) for k in DEFAULT_DATA_TYPES)
                                                     if u]
if len(SELECTED_TYPES) != 2:
    raise RuntimeError("Two data types are needed. Go back to the previous cell and confirm two.")

station_obj = stations_by_code[SELECTED_STATION]
STATION_URI_HTTP = station_obj.uri.replace("https://", "http://")
print(f"Station: {station_obj.id} - {station_obj.name}")


def _resolve_time_column(frame):
    """Products do not agree on the name: flask data use SamplingStart, ATC series TIMESTAMP."""
    for name in TIME_COLUMN_CANDIDATES:
        if name in frame.columns:
            return name
    for name in frame.columns:
        if pd.api.types.is_datetime64_any_dtype(frame[name]):
            return name
    return None


def load_one_type(datatype_uri):
    """Every data object of this type at this station, merged into one frame."""
    label = datatypes_by_uri[datatype_uri].label
    objs = meta.list_data_objects(datatype=datatype_uri, station=STATION_URI_HTTP)
    if not objs:
        print(f"  {label}: no data object at this station.")
        return None, []

    merged = None
    columns = []
    for dobj, arrs in data.batch_get_columns_as_arrays(objs[:MAX_OBJECTS_PER_TYPE]):
        d = pd.DataFrame(arrs)
        tcol = _resolve_time_column(d)
        if tcol is None:
            print(f"  {label}: no time column in {dobj.filename} - skipped.")
            continue
        if DEFAULT_QUALITY_FLAG in d.columns:
            d = d[d[DEFAULT_QUALITY_FLAG] == VALUE_ACCEPT_QUALITY]
        d = d.rename(columns={tcol: "time"})
        d["time"] = pd.to_datetime(d["time"], errors="coerce")

        height = f"_{dobj.sampling_height:g}m" if dobj.sampling_height else ""
        keep = {}
        for col in d.columns:
            if col == "time" or col in ("Flag", "NbPoints", "Stdev"):
                continue   # ancillary columns: not measurements, and they collide across types
            if pd.api.types.is_numeric_dtype(d[col]):
                keep[col] = f"{col}{height}"
        if not keep:
            continue
        part = d[["time"] + list(keep)].rename(columns=keep).dropna(subset=["time"])
        columns += list(keep.values())
        merged = part if merged is None else pd.merge(merged, part, on="time", how="outer")

    if merged is not None:
        print(f"  {label}: {len(objs)} object(s), {len(merged)} rows, columns {columns}")
    return merged, columns


print("Loading:")
frame_a, cols_a = load_one_type(SELECTED_TYPES[0])
frame_b, cols_b = load_one_type(SELECTED_TYPES[1])

if frame_a is None or frame_b is None:
    raise RuntimeError("One of the two data types has no data at this station. "
                       "Pick another type, or another station.")

def _describe(name, frame):
    print(f"  {name}: {len(frame)} rows, {frame['time'].min()} -> {frame['time'].max()}")
    print(f"     first timestamps: {[str(t) for t in frame['time'].head(3)]}")


combined = pd.merge(frame_a, frame_b, on="time", how="inner").sort_values("time")

if combined.empty:
    # An empty merge is usually not a missing period but timestamps that do not fall on the
    # same instant: different averaging intervals, or means labelled at the middle of the
    # interval rather than at its start.
    print("\nNo exactly matching timestamps. What the two records actually contain:")
    _describe(datatypes_by_uri[SELECTED_TYPES[0]].label, frame_a)
    _describe(datatypes_by_uri[SELECTED_TYPES[1]].label, frame_b)

    overlap = (max(frame_a["time"].min(), frame_b["time"].min()),
               min(frame_a["time"].max(), frame_b["time"].max()))
    if overlap[0] > overlap[1]:
        print("\nThe two periods do not overlap at all: nothing can be compared.")
    else:
        print(f"\nThe periods do overlap ({overlap[0]} -> {overlap[1]}), so the timestamps are "
              f"simply not aligned.")
        print(f"Matching each point with the nearest one within {ALIGN_TOLERANCE} ...")
        combined = pd.merge_asof(
            frame_a.sort_values("time"), frame_b.sort_values("time"),
            on="time", tolerance=pd.Timedelta(ALIGN_TOLERANCE), direction="nearest",
        ).dropna(subset=[c for c in cols_b])
        if combined.empty:
            print("Still nothing within that tolerance. Widen ALIGN_TOLERANCE in the "
                  "Configuration cell, or compare two products with the same sampling.")
        else:
            print(f"{len(combined)} points matched. Note that the two values of a pair are close "
                  f"in time but not simultaneous.")
else:
    print(f"\n{len(combined)} timestamps present in both records.")

## 6. Plot the two species on separate axes

Two different species rarely share a scale, so each goes on its own axis: the first on the left,
the second on the right.

In [ ]:
col_a = cols_a[0] if cols_a else None
col_b = cols_b[0] if cols_b else None

mode_dd = widgets.Dropdown(
    options=[("Both, on separate axes", "both"),
             ("Left series only", "left"),
             ("Right series only", "right")],
    value="both", description="Show:", style={"description_width": "initial"},
    layout=widgets.Layout(width="360px"))
series_a = widgets.Dropdown(options=cols_a, value=col_a, description="Left axis:",
                            style={"description_width": "initial"},
                            layout=widgets.Layout(width="420px"))
series_b = widgets.Dropdown(options=cols_b, value=col_b, description="Right axis:",
                            style={"description_width": "initial"},
                            layout=widgets.Layout(width="420px"))
plot_btn = widgets.Button(description="Plot", button_style="success", icon="line-chart")
plot_out = widgets.Output()


def _draw(_=None):
    with plot_out:
        plot_out.clear_output()
        if combined.empty:
            print("Nothing to plot: the two records have no timestamps in common.")
            return

        mode = mode_dd.value
        # A single series does not need a second axis, and drawing one would leave an empty
        # scale on the right; the two-axis layout is only built when both are shown.
        series_a.disabled = mode == "right"
        series_b.disabled = mode == "left"

        fig, ax1 = plt.subplots(figsize=(13, 5))
        ax1.set_xlabel("Time")

        if mode in ("both", "left"):
            ax1.plot(combined["time"], combined[series_a.value], linewidth=0.8, color="tab:blue")
            ax1.set_ylabel(series_a.value, color="tab:blue" if mode == "both" else "black")
            if mode == "both":
                ax1.tick_params(axis="y", labelcolor="tab:blue")

        if mode == "right":
            ax1.plot(combined["time"], combined[series_b.value], linewidth=0.8, color="tab:red")
            ax1.set_ylabel(series_b.value)
        elif mode == "both":
            ax2 = ax1.twinx()
            ax2.plot(combined["time"], combined[series_b.value], linewidth=0.8, color="tab:red")
            ax2.set_ylabel(series_b.value, color="tab:red")
            ax2.tick_params(axis="y", labelcolor="tab:red")

        shown = {"both": f"{series_a.value} and {series_b.value}",
                 "left": series_a.value, "right": series_b.value}[mode]
        ax1.set_title(f"{station_obj.name}: {shown}")
        fig.tight_layout()
        register_figure(fig, f"{SELECTED_STATION}_{shown.replace(' ', '_')}")
        plt.show()



for w in (mode_dd, series_a, series_b):
    w.observe(_draw, names="value")
plot_btn.on_click(_draw)
display(widgets.VBox([mode_dd, series_a, series_b, plot_btn, plot_out]))
_draw()

<a id="export"></a>
## Keeping the results

The figure is registered as it is drawn. Run the cell below to write it out, together with the
merged table, into an `outputs/` folder next to this notebook.

In [ ]:
# OUTPUT_DIR and exports/ are created and emptied by the setup cell at the top, as in the
# other notebooks of this VRE; here we only write into them.
written = []
for stem, fig in DRAWN_FIGURES.items():
    p = OUTPUT_DIR / f"{stem}.png"
    fig.savefig(p, dpi=150, bbox_inches="tight")
    written.append(p.name)

csv_path = OUTPUT_DIR / f"{SELECTED_STATION}_two_species.csv"
combined.to_csv(csv_path, index=False)

print(f"Figures: {', '.join(written) if written else 'none drawn yet'}")
print(f"Table  : {csv_path} ({len(combined)} rows)")
print("Download them from the file browser on the left: right-click, then Download.")